In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor




In [2]:
base_path = "../input/osic-pulmonary-fibrosis-progression/"
train_path = os.path.join(base_path, "train.csv")
test_path = os.path.join(base_path, "test.csv")
sample_sub_path = os.path.join(base_path, "sample_submission.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sub_df = pd.read_csv(sample_sub_path)




In [3]:
def add_features(df):
    base_week = df.groupby("Patient")["Weeks"].min()
    df["base_week"] = df["Patient"].map(base_week)

    df["count_from_base_week"] = df["Weeks"] - df["base_week"]

    base_fvc = df.groupby("Patient").apply(
        lambda g: g.loc[g["Weeks"] == g["Weeks"].min(), "FVC"].iloc[0]
    )
    df["base_fvc"] = df["Patient"].map(base_fvc)

    def fev1_est(row):
        A = row["base_fvc"]
        B = row["Age"]
        if row["Sex"] == "Male":
            return 0.77 * A + 0.32 + 0.0069 * B
        else:
            return 0.77 * A + 0.28 + 0.0052 * B

    df["base_fev1"] = df.apply(fev1_est, axis=1)

    df["base_height"] = (df["base_fvc"] + 9030) / 77.0
    df["base_weight"] = np.where(
        df["Sex"] == "Male",
        (df["base_fvc"] + 5458 - 49 * df["base_height"] + 8 * df["Age"]) / 12.0,
        (df["base_fvc"] + 3863 - 37 * df["base_height"] + 6 * df["Age"]) / 14.0,
    )
    df["base_bmi"] = df["base_weight"] / ((df["base_height"] / 100) ** 2)

    df["Weeks_sq"] = df["Weeks"] ** 2
    df["Age_sq"] = df["Age"] ** 2
    return df


train_df = add_features(train_df)
test_df = add_features(test_df)  # use its own stats for baseline mapping




/tmp/ipykernel_55/3011152087.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  base_fvc = df.groupby("Patient").apply(
/tmp/ipykernel_55/3011152087.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  base_fvc = df.groupby("Patient").apply(


In [4]:
le_sex = LabelEncoder()
le_smoke = LabelEncoder()
le_sex.fit(pd.concat([train_df["Sex"], test_df["Sex"]]))
le_smoke.fit(pd.concat([train_df["SmokingStatus"], test_df["SmokingStatus"]]))

train_df["Sex_enc"] = le_sex.transform(train_df["Sex"])
train_df["Smoking_enc"] = le_smoke.transform(train_df["SmokingStatus"])
test_df["Sex_enc"] = le_sex.transform(test_df["Sex"])
test_df["Smoking_enc"] = le_smoke.transform(test_df["SmokingStatus"])

for df in [train_df, test_df]:
    df["Weeks_Age_inter"] = df["Weeks"] * df["Age"]
    df["Weeks_Sex_inter"] = df["Weeks"] * df["Sex_enc"]
    df["Age_Sex_inter"] = df["Age"] * df["Sex_enc"]




In [5]:
feature_cols = [
    "Weeks",
    "Age",
    "Sex_enc",
    "Smoking_enc",
    "base_week",
    "count_from_base_week",
    "base_fvc",
    "base_fev1",
    "Percent",
    "base_height",
    "base_bmi",
    "base_weight",
    "Weeks_sq",
    "Age_sq",
    "Weeks_Age_inter",
    "Weeks_Sex_inter",
    "Age_Sex_inter",
]

X = train_df[feature_cols].values
y = train_df["FVC"].values

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(
    n_estimators=1500,
    max_depth=35,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=5,
)
rf.fit(X_train, y_train)


def laplace_metric(y_true, y_pred, confidence):
    sigma = np.maximum(confidence, 70)
    delta = np.minimum(np.abs(y_true - y_pred), 1000)
    score = -np.sqrt(2) * delta / sigma - np.log(np.sqrt(2) * sigma)
    return np.mean(score)


val_pred = rf.predict(X_valid)
val_residuals = np.abs(y_valid - val_pred)
mean_residual = val_residuals.mean()

candidate_factors = [
    2.5,
    3.0,
    3.5,
    4.0,
    5.0,
    6.0,
    8.0,
    10.0,
    12.0,
    15.0,
    20.0,
    30.0,
    40.0,
    50.0,
]

best_factor = None
best_score = -np.inf
# ---- evaluate candidates with per‑sample σ ----
for factor in candidate_factors:
    sigma_vec = np.maximum(70, val_residuals * factor)
    score = laplace_metric(y_valid, val_pred, sigma_vec)
    if score > best_score:
        best_score = score
        best_factor = factor

fine_factors = np.arange(max(1.0, best_factor - 1.0), best_factor + 1.0 + 0.1, 0.1)
for factor in fine_factors:
    sigma_vec = np.maximum(70, val_residuals * factor)
    score = laplace_metric(y_valid, val_pred, sigma_vec)
    if score > best_score:
        best_score = score
        best_factor = factor

extra_factors = np.arange(best_factor + 1, 1001, 1)  # up to 1000
for factor in extra_factors:
    sigma_vec = np.maximum(70, val_residuals * factor)
    score = laplace_metric(y_valid, val_pred, sigma_vec)
    if score > best_score:
        best_score = score
        best_factor = factor

# constant confidence used for test predictions (kept compatible with submission format)
derived_confidence = max(70, mean_residual * best_factor)
val_score = laplace_metric(y_valid, val_pred, np.full_like(y_valid, derived_confidence))

print(f"Chosen confidence multiplier: {best_factor:.2f}")
print(
    f"Validation Laplace Log Likelihood (σ={derived_confidence:.1f}): {val_score:.5f}"
)




Chosen confidence multiplier: 1.50
Validation Laplace Log Likelihood (σ=113.1): -6.01764


In [6]:
sub_df[["Patient", "WeekIdx"]] = sub_df["Patient_Week"].str.rsplit(
    "_", n=1, expand=True
)
sub_df["WeekIdx"] = sub_df["WeekIdx"].astype(int)

test_meta = test_df.drop(columns=["Weeks"])
test_expanded = sub_df[["Patient", "WeekIdx"]].merge(
    test_meta, on="Patient", how="left"
)

test_expanded.rename(columns={"WeekIdx": "Weeks"}, inplace=True)

test_expanded["count_from_base_week"] = (
    test_expanded["Weeks"] - test_expanded["base_week"]
)

test_expanded["Weeks_sq"] = test_expanded["Weeks"] ** 2
test_expanded["Age_sq"] = test_expanded["Age"] ** 2

test_expanded["Weeks_Age_inter"] = test_expanded["Weeks"] * test_expanded["Age"]
test_expanded["Weeks_Sex_inter"] = test_expanded["Weeks"] * test_expanded["Sex_enc"]
test_expanded["Age_Sex_inter"] = test_expanded["Age"] * test_expanded["Sex_enc"]

X_test = test_expanded[feature_cols].values
test_pred_fvc = rf.predict(X_test)

test_confidence = np.full_like(test_pred_fvc, derived_confidence)

submission = pd.DataFrame(
    {
        "Patient_Week": sub_df["Patient_Week"],
        "FVC": test_pred_fvc,
        "Confidence": test_confidence,
    }
)




In [7]:
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")

Submission saved to submission.csv
